# Post-hoc benchmark: pretrained LRW MS-TCN on the already viewed final test

This notebook evaluates the selected pretrained MS-TCN on the existing final-test
archive. The archive was previously used for the BiGRU benchmark, so these numbers
are a post-hoc benchmark and must not be used to tune the model.

## 1. Connect Drive and restore fixed data/checkpoint

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
%pip -q install gdown

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import time
import zipfile
import gdown
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader


def load_torch_file(path, device='cpu'):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


drive_root = Path('/content/drive/MyDrive/lipreading_sem4')
train_extract = Path('/content/lipreading_data_v3')
test_extract = Path('/content/lipreading_final_test_landmark_v3')
output_dir = drive_root / 'posthoc_mstcn_final_test_benchmark'
output_dir.mkdir(parents=True, exist_ok=True)

upstream_dir = Path('/content/Lipreading_using_Temporal_Convolutional_Networks')
if not upstream_dir.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/mpc001/Lipreading_using_Temporal_Convolutional_Networks.git',
        str(upstream_dir),
    ], check=True)
sys.path.insert(0, str(upstream_dir))

checkpoint_path = drive_root / 'upstream_lrw_pretrained' / 'lrw_resnet18_mstcn_video.pth'
if not checkpoint_path.exists():
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    if gdown.download(id='1vqMpxZ5LzJjg50HlZdj_QFJGm2gQmDUD', output=str(checkpoint_path), quiet=False) is None:
        raise RuntimeError('Official LRW checkpoint download failed.')

train_split = train_extract / 'ru_dataset' / '03_splits' / 'ml_splits_mouth_crops_landmark_dataset_v3' / 'speaker_top10'
if not (train_split / 'train.csv').exists():
    matches = list(Path('/content/drive').rglob('colab_lipreading_dataset_v3_train_val.zip'))
    if not matches:
        raise FileNotFoundError('Upload colab_lipreading_dataset_v3_train_val.zip to Google Drive.')
    if train_extract.exists(): shutil.rmtree(train_extract)
    with zipfile.ZipFile(matches[0]) as archive: archive.extractall(train_extract)

test_split = test_extract / 'ru_dataset' / '03_splits' / 'final_test_landmark_v3'
if not (test_split / 'test.csv').exists():
    matches = list(Path('/content/drive').rglob('colab_lipreading_final_test_landmark_v3.zip'))
    if not matches:
        raise FileNotFoundError('Upload colab_lipreading_final_test_landmark_v3.zip to Google Drive.')
    if test_extract.exists(): shutil.rmtree(test_extract)
    with zipfile.ZipFile(matches[0]) as archive: archive.extractall(test_extract)

train_df = pd.read_csv(train_split / 'train.csv')
test_df = pd.read_csv(test_split / 'test.csv')
vocab = (train_split / 'vocab.txt').read_text(encoding='utf-8').splitlines()
assert (test_split / 'vocab.txt').read_text(encoding='utf-8').splitlines() == vocab
word_to_idx = {word: index for index, word in enumerate(vocab)}
idx_to_word = {index: word for word, index in word_to_idx.items()}
feature_root = drive_root / 'controlled_lrw_frontend_bigru_dataset_v3'
train_original = load_torch_file(feature_root / 'dataset_v3_train_lrw_mstcn_pooled.pt')['features'].float()
train_hflip = load_torch_file(feature_root / 'dataset_v3_train_lrw_mstcn_hflip_pooled.pt')['features'].float()
assert train_original.shape == train_hflip.shape == (len(train_df), 768)
assert list(train_df['word']) == train_df['word'].tolist()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda': raise RuntimeError('Enable a GPU runtime before running this benchmark.')
print('train:', len(train_df), 'test:', len(test_df), 'vocab:', vocab)
print('train pooled features:', train_original.shape)
print('device:', device)
print('WARNING: this is a post-hoc benchmark on the already viewed final test.')

## 2. Extract fixed pretrained MS-TCN features for final-test

In [ ]:
import cv2
from lipreading.model import Lipreading

def load_lrw_video(path, num_frames=24, input_size=96, crop_size=88):
    cap = cv2.VideoCapture(str(path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise ValueError(f'Cannot read video: {path}')
    frames = []
    for frame_index in np.linspace(0, total_frames - 1, num_frames).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, frame = cap.read()
        if not ok: continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame = cv2.resize(frame, (input_size, input_size))
        offset = (input_size - crop_size) // 2
        frame = frame[offset:offset + crop_size, offset:offset + crop_size]
        frame = frame.astype(np.float32) / 255.0
        frames.append((frame - 0.421) / 0.165)
    cap.release()
    if not frames: raise ValueError(f'No frames loaded: {path}')
    while len(frames) < num_frames: frames.append(frames[-1].copy())
    return torch.from_numpy(np.stack(frames)).float().unsqueeze(0)

class TestVideoDataset(Dataset):
    def __len__(self): return len(test_df)
    def __getitem__(self, index):
        row = test_df.iloc[index]
        return load_lrw_video(test_extract / 'ru_dataset' / row['clip_path']), index

test_video_loader = DataLoader(TestVideoDataset(), batch_size=4, shuffle=False, num_workers=2)
tcn_options = {'num_layers': 4, 'kernel_size': [3, 5, 7], 'dropout': 0.2, 'dwpw': False, 'width_mult': 1}
pretrained_model = Lipreading(modality='video', num_classes=500, tcn_options=tcn_options, backbone_type='resnet', relu_type='swish', width_mult=1.0, use_boundary=False, extract_feats=True).to(device)
checkpoint = load_torch_file(checkpoint_path)
pretrained_model.load_state_dict(checkpoint.get('model_state_dict', checkpoint), strict=True)
for parameter in pretrained_model.parameters(): parameter.requires_grad = False
pretrained_model.eval()

test_cache_path = output_dir / 'final_test_mstcn_pooled_features.pt'
if test_cache_path.exists():
    test_tcn = load_torch_file(test_cache_path)['features'].float()
    print('loaded cached test MS-TCN features')
else:
    batches = []
    with torch.inference_mode():
        for batch_index, (videos, _) in enumerate(test_video_loader, start=1):
            sequence = pretrained_model(videos.to(device), lengths=[videos.shape[2]] * videos.shape[0])
            pooled = pretrained_model.tcn.mb_ms_tcn(sequence.transpose(1, 2)).mean(dim=2)
            batches.append(pooled.cpu())
            if batch_index == 1 or batch_index % 20 == 0: print(f'feature batches: {batch_index}/{len(test_video_loader)}')
    test_tcn = torch.cat(batches)
    torch.save({'features': test_tcn, 'words': test_df['word'].tolist(), 'speakers': test_df['speaker_id'].tolist(), 'test_used_for_training': False}, test_cache_path)
assert test_tcn.shape == (len(test_df), 768)
print('test pooled features:', test_tcn.shape)

In [ ]:
## 3. Train a fixed linear head on train v3 and evaluate post-hoc on final-test

class VectorDataset(Dataset):
    def __init__(self, features, words):
        self.features = features.float()
        self.labels = torch.tensor([word_to_idx[word] for word in words], dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, index): return self.features[index], self.labels[index]

class LinearProbe(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.classifier = nn.Linear(input_dim, num_classes)
    def forward(self, features): return self.classifier(features)

def set_seed(seed):
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def metrics_for(preds, labels):
    matrix = np.zeros((len(vocab), len(vocab)), dtype=int)
    for true_id, pred_id in zip(labels, preds): matrix[true_id, pred_id] += 1
    rows = []
    for class_id, word in enumerate(vocab):
        tp = int(matrix[class_id, class_id]); support = int(matrix[class_id].sum()); predicted = int(matrix[:, class_id].sum())
        precision = tp / predicted if predicted else 0.0; recall = tp / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({'class_id': class_id, 'word': word, 'support': support, 'predicted_count': predicted, 'precision': precision, 'recall': recall, 'f1': f1})
    return {'accuracy': float(np.trace(matrix) / matrix.sum()), 'macro_f1': float(np.mean([row['f1'] for row in rows])), 'balanced_accuracy': float(np.mean([row['recall'] for row in rows])), 'predicted_classes': int(np.count_nonzero(matrix.sum(axis=0))), 'class_rows': rows, 'confusion_matrix': matrix}

train_x = torch.cat([train_original, train_hflip])
train_words = train_df['word'].tolist() * 2
test_y = torch.tensor([word_to_idx[word] for word in test_df['word']], dtype=torch.long)
train_loader = DataLoader(VectorDataset(train_x, train_words), batch_size=64, shuffle=True, num_workers=0)
runs = []; metrics_by_seed = {}
for seed in [42, 123, 2026]:
    set_seed(seed)
    model = LinearProbe(768, len(vocab)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(10):
        model.train()
        for features, labels in train_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(features.to(device)), labels.to(device))
            loss.backward(); optimizer.step()
    model.eval(); preds = []
    with torch.no_grad():
        for start in range(0, len(test_tcn), 64):
            preds.extend(model(test_tcn[start:start + 64].to(device)).argmax(1).cpu().tolist())
    labels = test_y.tolist(); metrics = metrics_for(preds, labels); metrics_by_seed[seed] = metrics
    runs.append({'seed': seed, 'test_accuracy': metrics['accuracy'], 'test_macro_f1': metrics['macro_f1'], 'test_balanced_accuracy': metrics['balanced_accuracy'], 'predicted_classes': metrics['predicted_classes'], 'test_clips': len(labels), 'test_speakers': int(test_df['speaker_id'].nunique()), 'epochs': 10})
    pd.DataFrame({'sample_index': range(len(labels)), 'speaker_id': test_df['speaker_id'], 'true_word': [idx_to_word[i] for i in labels], 'predicted_word': [idx_to_word[i] for i in preds]}).to_csv(output_dir / f'posthoc_mstcn_test_predictions_seed{seed}.csv', index=False)
    pd.DataFrame(metrics['class_rows']).to_csv(output_dir / f'posthoc_mstcn_test_per_class_seed{seed}.csv', index=False)
    print(f"seed={seed} | acc={metrics['accuracy']:.4f} | macro-F1={metrics['macro_f1']:.4f} | balanced-acc={metrics['balanced_accuracy']:.4f} | classes={metrics['predicted_classes']}/10")

runs_df = pd.DataFrame(runs).sort_values('seed')
runs_df.to_csv(output_dir / 'posthoc_mstcn_final_test_runs.csv', index=False)
display(runs_df)

In [ ]:
## 4. Save summary and freeze interpretation

summary = runs_df[['test_accuracy', 'test_macro_f1', 'test_balanced_accuracy', 'predicted_classes']].agg(['mean', 'std']).T
summary.columns = ['mean', 'std']; summary.reset_index(names='metric').to_csv(output_dir / 'posthoc_mstcn_final_test_summary.csv', index=False)
per_class = []
combined_matrix = np.zeros((len(vocab), len(vocab)), dtype=int)
for seed, metrics in metrics_by_seed.items():
    frame = pd.DataFrame(metrics['class_rows']); frame['seed'] = seed; per_class.append(frame); combined_matrix += metrics['confusion_matrix']
pd.concat(per_class, ignore_index=True).groupby('word')[['precision', 'recall', 'f1']].agg(['mean', 'std']).to_csv(output_dir / 'posthoc_mstcn_final_test_per_class_summary.csv')
print(summary)
print('Results saved to:', output_dir)
print('POST-HOC BENCHMARK COMPLETED. Do not tune the model using these test metrics.')

In [ ]:
class BiGRUClassifier(nn.Module):
    def __init__(self, num_classes, input_dim=512, hidden_dim=128, dropout=0.4):
        super().__init__()
        self.temporal_encoder = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, num_classes),
        )

    def forward(self, features):
        _, hidden = self.temporal_encoder(features)
        clip_features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(clip_features)


class TestFeatureDataset(Dataset):
    def __init__(self):
        self.features = test_cache['features'].float()
        self.labels = torch.tensor(
            [word_to_idx[word] for word in test_cache['words']], dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


test_loader = DataLoader(
    TestFeatureDataset(), batch_size=32, shuffle=False, num_workers=0
)


def classification_metrics(preds, labels, num_classes):
    matrix = np.zeros((num_classes, num_classes), dtype=int)
    for true_index, pred_index in zip(labels, preds):
        matrix[true_index, pred_index] += 1

    class_rows = []
    for class_id in range(num_classes):
        tp = int(matrix[class_id, class_id])
        support = int(matrix[class_id].sum())
        predicted = int(matrix[:, class_id].sum())
        precision = tp / predicted if predicted else 0.0
        recall = tp / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        class_rows.append({
            'class_id': class_id,
            'word': idx_to_word[class_id],
            'support': support,
            'predicted_count': predicted,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        })

    active = [row for row in class_rows if row['support']]
    return {
        'accuracy': float(np.trace(matrix) / matrix.sum()),
        'macro_f1': float(np.mean([row['f1'] for row in active])),
        'balanced_accuracy': float(np.mean([row['recall'] for row in active])),
        'predicted_classes': int(np.count_nonzero(matrix.sum(axis=0))),
        'class_rows': class_rows,
        'confusion_matrix': matrix,
    }


def evaluate_checkpoint(seed):
    saved = load_torch_file(model_paths[seed])
    assert saved['vocab'] == vocab
    model = BiGRUClassifier(len(vocab)).to(device)
    model.load_state_dict(saved['model_state_dict'], strict=True)
    model.eval()

    preds = []
    labels = []
    with torch.no_grad():
        for features, targets in test_loader:
            outputs = model(features.to(device))
            preds.extend(outputs.argmax(dim=1).cpu().tolist())
            labels.extend(targets.tolist())

    metrics = classification_metrics(preds, labels, len(vocab))
    row = {
        'seed': seed,
        'test_accuracy': metrics['accuracy'],
        'test_macro_f1': metrics['macro_f1'],
        'test_balanced_accuracy': metrics['balanced_accuracy'],
        'predicted_classes': metrics['predicted_classes'],
        'test_clips': len(labels),
        'test_speakers': 3,
    }
    prediction_table = pd.DataFrame({
        'sample_index': range(len(labels)),
        'speaker_id': test_cache['speakers'],
        'true_word': [idx_to_word[index] for index in labels],
        'predicted_word': [idx_to_word[index] for index in preds],
    })
    prediction_table.to_csv(final_dir / f'test_predictions_seed{seed}.csv', index=False)
    pd.DataFrame(metrics['class_rows']).to_csv(
        final_dir / f'test_per_class_seed{seed}.csv', index=False
    )
    return row, metrics


print('Evaluation code ready. There is no optimizer or training loop.')

## 5. Единственное финальное test evaluation

In [ ]:
rows = []
metrics_by_seed = {}
for seed in seeds:
    row, metrics = evaluate_checkpoint(seed)
    rows.append(row)
    metrics_by_seed[seed] = metrics
    print(
        f"seed={seed} | test acc={row['test_accuracy']:.4f} | "
        f"macro-F1={row['test_macro_f1']:.4f} | "
        f"balanced_acc={row['test_balanced_accuracy']:.4f} | "
        f"classes={row['predicted_classes']}/10"
    )

test_runs = pd.DataFrame(rows).sort_values('seed')
test_runs.to_csv(final_dir / 'untouched_final_test_multiseed_runs.csv', index=False)
display(test_runs)

## 6. Финальная таблица, per-class recall и confusion matrix

In [ ]:
metric_columns = [
    'test_accuracy',
    'test_macro_f1',
    'test_balanced_accuracy',
    'predicted_classes',
]
summary = test_runs[metric_columns].agg(['mean', 'std']).T
summary.columns = ['mean', 'std']
summary.reset_index(names='metric').to_csv(
    final_dir / 'untouched_final_test_multiseed_summary.csv', index=False
)
display(summary)

per_class_frames = []
combined_matrix = np.zeros((len(vocab), len(vocab)), dtype=int)
for seed, metrics in metrics_by_seed.items():
    frame = pd.DataFrame(metrics['class_rows'])
    frame['seed'] = seed
    per_class_frames.append(frame)
    combined_matrix += metrics['confusion_matrix']

per_class_all = pd.concat(per_class_frames, ignore_index=True)
per_class_summary = per_class_all.groupby('word')[['precision', 'recall', 'f1']].agg(['mean', 'std'])
per_class_summary.to_csv(final_dir / 'untouched_final_test_per_class_summary.csv')
display(per_class_summary)

fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(combined_matrix, cmap='Blues')
ax.set_title('Untouched final test confusion matrix, summed across three seeds')
ax.set_xlabel('predicted word')
ax.set_ylabel('true word')
ax.set_xticks(range(len(vocab)), vocab, rotation=45, ha='right')
ax.set_yticks(range(len(vocab)), vocab)
fig.colorbar(image, ax=ax)
fig.tight_layout()
fig.savefig(final_dir / 'untouched_final_test_confusion_matrix_three_seeds.png', dpi=150)
plt.show()

print('UNTOUCHED FINAL TEST COMPLETED.')
print('Freeze these results. Do not tune the model using test metrics.')
print('Results saved to:', final_dir)